For a different approach: https://en.wikipedia.org/wiki/Newell%27s_algorithm

Case we do not cover: One face a subset of another face.

This will never happen for a CP.

Maybe test for it in the beginning

In [ ]:
%pylab inline
import eucare as ec

render_settings = dict(
    figsize=(14, 14),
    #scale=100,
    render_edges=False,
    render_faces=True,
    render_vertices=False,
    line_width=0.0,
    face_inset=0.0,
    for_cutting = False
)

In [ ]:
def rotate_by(list_like, offset):
    if isinstance(offset, int):
        l = list(list_like)
        return l[offset:] + l[:offset]
    else:
        return zip(*(rotate_by(list_like, off) for off in offset))


def intervals_overlapping(interval1, interval2):
    """
    Checks if two intervals are overlapping
    
    Parameters
    ----------
    interval1 : tuple
    interval2 : tuple
    
    Returns
    -------
    bool
    
    Examples
    --------
    >>> intervals_overlapping([0, 1], [-1, 0.5])
    True
    >>> intervals_overlapping([0, 1], [2, 3])
    False
    >>> intervals_overlapping([0, 1], [1, 2])
    True
    
    """
    return not (interval1[0] > interval2[1]) and not (interval1[1] < interval2[0])

def get_potential_intersections(segments, epsilon=1e-12):
    # list of start and end points, with index of corresponding segment and flag whether it is a start or an end point.
    segments = np.array(segments).copy()
    segments.sort(axis=1)
    segments[:, 0, :] -= epsilon 
    print(segments.shape)
    assert len(segments.shape) == 3, f'{segments.shape}'
    assert segments.shape[1:] == (2, 2), f'{segments.shape}'
    x_coords = segments[:, :, 0]
    x_coords.sort(axis=1)
    points = [(s[0][0] - epsilon, i, 1) for i, s in enumerate(segments)] + [(s[1][0], i, 0) for i, s in enumerate(segments)]
    points = np.array(points, dtype=tuple)
    points = points[np.argsort(points[:, 0])]
    active_labels = set()
    possibly_intersecting = list()
    for i, is_start in points[:, 1:]:
        if is_start:
            for j in active_labels:
                if intervals_overlapping(segments[i, :, 1], segments[j, :, 1]):
                    possibly_intersecting.append((i, j))
            active_labels.add(i)
        else:
            active_labels.remove(i)
    return possibly_intersecting

def _on_segment(p, q, r):
        """Checks if q lies on segment pr. Points are assumed to be collinear."""
        return (min(p[0], r[0]) <= q[0] <= max(p[0], r[0])) and (min(p[1], r[1]) <= q[1] <= max(p[1], r[1]))

def _det(a, b):
        return a[0] * b[1] - a[1] * b[0]
    
def line_segment_intersections(s1, s2, eps=1e-12):
    """
    see https://www.geeksforgeeks.org/check-if-two-given-line-segments-intersect/
    """
    
#     if not intervals_overlapping([min(s1[:, 0]), max(s1[:, 0])], [min(s2[:, 0]), max(s2[:, 0])]):
#         return []  # separated in x
#     if not intervals_overlapping([min(s1[:, 1]), max(s1[:, 1])], [min(s2[:, 1]), max(s2[:, 1])]):
#         return []  # separated in y
    
    p1, q1 = s1
    p2, q2 = s2
    o1 = ec.base.orientation([p1, q1, p2], eps) 
    o2 = ec.base.orientation([p1, q1, q2], eps) 
    o3 = ec.base.orientation([p2, q2, p1], eps) 
    o4 = ec.base.orientation([p2, q2, q1], eps)
    
    # General case 
    if (o1 != o2 and o3 != o4):
        xdiff = (p1[0] - q1[0], p2[0] - q2[0])
        ydiff = (p1[1] - q1[1], p2[1] - q2[1])
        div = _det(xdiff, ydiff)
        if div == 0:
            #return [] # TODO: investigate when this happens..
            raise Exception('lines do not intersect')
        d = (_det(p1, q1), _det(p2, q2))
        x = _det(d, xdiff) / div
        y = _det(d, ydiff) / div
        return [[x, y]]
    
    # Special Cases (or no intersection)
    result = []
    
    # p1, q1 and p2 are colinear and p2 lies on segment p1q1 
    if (o1 == 0 and _on_segment(p1, p2, q1)):
        result.append(p2)
  
    # p1, q1 and q2 are colinear and q2 lies on segment p1q1 
    if (o2 == 0 and _on_segment(p1, q2, q1)):
        result.append(q2)
  
    # p2, q2 and p1 are colinear and p1 lies on segment p2q2 
    if (o3 == 0 and _on_segment(p2, p1, q2)):
        result.append(p1)
  
    # p2, q2 and q1 are colinear and q1 lies on segment p2q2 
    if (o4 == 0 and _on_segment(p2, q1, q2)):
        result.append(q1)
        
    return result
    
def poly_intersection_points(p1, p2):
    """
    Compute points of intersection bewteen two 2D polygons.
    
    Parameters
    ----------
    p1 : np.ndarray
    ccw points in first polygon.
    p2 : np.ndarray
    ccw points in second polygon.
    
    Returns
    -------
    np.ndarray or False
        List of intersection points. If none are found, returns False.
        
    """
    
    if not intervals_overlapping([np.min(p1[:, 0]), np.max(p1[:, 0])], [np.min(p1[:, 0]), np.max(p1[:, 0])]):
        return False  # separated in x
    if not intervals_overlapping([np.min(p1[:, 1]), np.max(p1[:, 1])], [np.min(p1[:, 1]), np.max(p1[:, 1])]):
        return False  # separated in y
    
    intersection_info = [(i, j, p) 
                           for i, s1 in enumerate(zip(p1, rotate_by(p1, 1)))
                           for j, s2 in enumerate(zip(p2, rotate_by(p2, 1)))
                           for p in line_segment_intersections[s1, s2]]
    return intersection_info

In [ ]:
import time

def transform_heg(G, transformation):
    assert isinstance(G, ec.half.EuclideanPositionHEG), f'{type(G)}'
    if G.order == 0:
        return
    p = G.get_position_view()[0]
    p[:] = transformation(p)
    
def n_gon_graph(n):
    return ec.half.EuclideanPositionHEG(
        other=ec.prototiles.RegularEuclideanTile(n).make_graph(add_positions=True)[0])
def cyclic_graph(points):
    return ec.half.EuclideanPositionHEG(
        other=ec.prototiles.EuclideanProtoTile(points).make_graph(add_positions=True)[0])

G = n_gon_graph(5)

transform_heg(G, lambda p: p * [8, 0.2] + [0, 0])

G.add_graph(n_gon_graph(4))

transform_heg(G, lambda p: p * [0.5, 3] + [0, 0])

G.add_graph(cyclic_graph(points=(np.array([[3, -2], [1, 3], [-0.8, 1.7]]))))

G.check_consistency()

transform_heg(G, lambda p: p * 0.4 + [0.5, 0.7])
G.add_graph(n_gon_graph(6))

G = ec.half.EuclideanPositionHEG()
for i in range(5):
    for j in range(5):
        v = [i/2, j/2]
        transform_heg(G, lambda p: p + v)
        G.add_graph(n_gon_graph(5))
        transform_heg(G, lambda p: p - v)

cc = ec.classifiers.congruency_classifier()
for f in G.faces:
    f['color_key'] = cc.classify(f)
            
G.show(**render_settings)

# ------------------------------------------
eps = 1e-10

def random_directed_set(edges):
    if isinstance(edges, ec.half.HalfEdgeGraph):
        edges = edges.halfedges
    directed_edges = set()
    for e in edges:
        if e.rev not in directed_edges:
            directed_edges.add(e)
    return directed_edges

edges = [e for e in G.halfedges if e.face is not None]
print('number of edges:', len(edges))
line_segments = np.array([[e.orig['pos'], e.dest['pos']] for e in edges])
print(line_segments.shape)

# ------ get all crossings ------
# TODO implement sweeping line algorithm https://www.geeksforgeeks.org/given-a-set-of-line-segments-find-if-any-two-segments-intersect/

crossing_dict = {i: {} for i in range(len(line_segments))}
crossings = []
crossings_to_edges = []
# crossing_dict[i][j] is a list of points where edges[i] and edges[j] cross.
# iterate over all pairs

potential_intersections = get_potential_intersections(line_segments, epsilon=eps)
#print('potential_intersections:', potential_intersections)
for i, j in potential_intersections:
    l1, l2 = line_segments[i], line_segments[j]
    #if any([e.nex in [edges[j], edges[j].rev] for e in [edges[i], edges[i].rev]]):
    #    continue
    intersections = line_segment_intersections(l1, l2, eps=eps)
    if not intersections:
        continue
    # if len(intersections) > 1:
    #     print('oho', i, j, intersections)
    crossings.extend(intersections)
    for _ in range(len(intersections)):
        crossings_to_edges.append((i, j))
    crossing_indices = list(range(len(crossings)-len(intersections), len(crossings)))
    for k, l in [(i, j), (j, i)]: # add intersections to both lines
        if l not in crossing_dict[k]:
            crossing_dict[k][l] = []
        crossing_dict[k][l].extend(crossing_indices)
crossings = np.array(crossings)

# ------ group closeby crossings ------

print('n crossings:', len(crossings))
from scipy.cluster import hierarchy
from sklearn.cluster import AgglomerativeClustering
clusterer = AgglomerativeClustering(n_clusters=None, distance_threshold=eps, linkage='single')
clusterer.fit(crossings)
clustering = clusterer.labels_
print('reduced n crossings', np.max(clustering)+1)
first_occurences = np.argmax(clustering[None] == np.arange(np.max(clustering)+1)[:, None], axis=1)
filtered_crossings = crossings[first_occurences]


n_filtered_crossings = len(filtered_crossings)
filtered_crossings_to_edges = [set() for i in range(n_filtered_crossings)]
edges_to_crossings = [set() for i in range(len(edges))]
for i, edge_ids in enumerate(crossings_to_edges):
    filtered_crossings_to_edges[clustering[i]].update(edge_ids)
    for e in edge_ids:
        edges_to_crossings[e].add(clustering[i])
    
# ------ get crossing orders, construct nx graph ------
import networkx as nx

#nodes = np.arange(n_filtered_crossings)
nx_edges = []
edge_to_ordered_ids = []
for i in range(len(line_segments)):
    e = edges[i]
    crossing_ids = np.array(list(edges_to_crossings[i]))
    crossing_positions = filtered_crossings[crossing_ids]
    progression_along_edge = ((crossing_positions - e.orig['pos'][None]) * (e.dest['pos'] - e.orig['pos'])[None]).sum(-1)
    order = np.argsort(progression_along_edge)
    ordered_ids = crossing_ids[order]
    edge_to_ordered_ids.append(ordered_ids)
    nx_edges.append(np.stack([ordered_ids[:-1], ordered_ids[1:]], axis=-1))
nx_edges = np.concatenate(nx_edges)
print(nx_edges.shape)

nx_graph = nx.Graph()
nx_graph.add_edges_from(nx_edges)
nx_positions = {i: pos for i, pos in enumerate(filtered_crossings)}

overlap_G, v_lookup = ec.conversions.EHEG_from_nx(nx_graph, nx_positions, return_v_lookup=True)
overlap_G.recompute_lengths_and_angles()
cc = ec.classifiers.congruency_classifier()
for f in overlap_G.faces:
    f['color_key'] = cc.classify(f)
overlap_G.show(**render_settings)

# ------ assign original vertices, edges ------
for v in overlap_G.vertices:
    v['original_vertices'] = set()
for e in overlap_G.halfedges:
    e['original_edges'] = set()
for f in overlap_G.faces:
    f['original_faces'] = set()
    
for i in range(len(edges)):
    ordered_ids = edge_to_ordered_ids[i]
    if len(ordered_ids) == 0:
        assert False, f'this should not happen..'
    for e in (edges[i], edges[i].rev):
        if e is edges[i].rev:
            ordered_ids = ordered_ids[::-1]
        v_lookup[ordered_ids[0]]['original_vertices'].add(e.orig)
        for k, l in zip(ordered_ids[:-1], ordered_ids[1:]):
            for new_edge in v_lookup[k].outgoing_iter():
                if v_lookup[l] is new_edge.dest:
                    new_edge['original_edges'].add(e)
                    
# ------ assign original faces ------
initial_edge = next(overlap_G.border_edge_iter()).rev
frontier = [(initial_edge, {e.face for e in initial_edge['original_edges']})]
yet_to_assign = set(overlap_G.faces)
current_original_faces = set()
while yet_to_assign:
    current_halfedge, original_faces = frontier.pop()
    if current_halfedge.on_border():
        assert False
    current_face = current_halfedge.face
    if current_face not in yet_to_assign:
        continue
    current_face['original_faces'] = original_faces
    yet_to_assign.remove(current_face)
    for e in current_face.halfedge_iter():
        if not e.rev.on_border() and e.rev.face in yet_to_assign:
            frontier.append((
                e.rev, 
                (original_faces - {e_orig.face for e_orig in e['original_edges'] if not e_orig.on_border()}).union({e_orig.face for e_orig in e.rev['original_edges'] if not e_orig.on_border()})
            ))
    
            
print('done')

In [ ]:
#IDEA: save 'face difference' along edges and get overlap faces from that by floodfilling.

In [ ]:
order = np.arange(len(G.faces))
np.random.shuffle(order)
for f in G.faces:
    f['height'] = np.random.rand()
    
for f in overlap_G.faces:
    overlapping_faces = list(f['original_faces'])
    if not overlapping_faces:
        assert False
    else:
        #print(overlapping_faces)
        f['color_key'] = len(overlapping_faces)
        f['top_face'] = overlapping_faces[np.argmax(np.array([fo['height'] for fo in overlapping_faces]))]


to_delete = [e 
             for e in overlap_G.halfedges
             if not (e.on_border() or e.rev.on_border()) and e.face['top_face'] is e.rev.face['top_face']]

# # does not work, because some edges cannot be deleted on their own
# for e in to_delete:
#         break
#     if e in overlap_G.halfedges:
#         overlap_G.delete_edge(e)

# ugly solution for now
overlap_G.halfedges.difference_update(to_delete)
final_G = ec.conversions.EHEG_from_nx(overlap_G.to_networkx_undirected(), {v: v['pos'] for v in overlap_G.vertices})
final_G.recompute_lengths_and_angles()
overlap_G.halfedges = overlap_G.halfedges.union(to_delete)

# final_G = overlap_G
# for e in final_G.halfedges:
#     e['delete'] = e in to_delete
    
cc = ec.classifiers.congruency_classifier()
#cc = ec.classifiers.CountingClassifier(ec.classifiers.lambda_classifier(lambda f: f['top_face'])())


for i, f in enumerate(final_G.faces):
    f['color_key'] = cc.classify(f)
    
final_G.show(**render_settings)

In [ ]:
from functools import reduce
set.union(set(), *(range(4*i, 4*i) for i in range(2, 5)))

In [ ]:
all([i for i in range(-2, 3)])